# C4 · QLoRA fine-tuning of Mistral 7B Instruct v0.3 — Email Agent
**Student:** Saayinath K (IT23304338) · **Component:** C4 · **Platform:** Google Colab (free T4 GPU)
Based on the C3-14 notebook by Shathursini J, changed for 3-step email sequences.

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save.* Then *Runtime → Run all*.
Put `train.jsonl` and `val.jsonl` (from `python prepare_email_ft.py`) in Google Drive → `MyDrive/c4_finetune/`
(or upload them when cell 2 asks).

**What is fine-tuned:** behaviour, not facts: valid JSON only, exactly 3 emails on days 0/3/7, the unsubscribe
line in every email, under 150 words, and **only** the prices / discounts / promo codes that are in the brand
context (no invented offers). Tenant facts still come from RAG (pgvector) at run time.

**Safe on a limited GPU:** everything is saved to Google Drive as it runs (base evaluation, checkpoints every
20 steps, adapter). If Colab disconnects, reconnect and *Run all* again: finished parts are skipped and
training resumes from the last checkpoint.

| Cell | What | Time on T4 |
| --- | --- | --- |
| 1 | install libraries | ~2 min |
| 2 | GPU check, Drive, data | ~1 min |
| 3 | download + load Mistral 7B in 4-bit | ~5 min |
| 4 | tokenise data, mask the prompt | <1 min |
| 5–6 | evaluation helpers + **before** evaluation (25 prompts) | ~15–20 min |
| 7 | QLoRA training (2 epochs, ~180 steps) | ~1.5–2 h |
| 8 | **after** evaluation + comparison | ~15–20 min |
| 9 | save adapter + downloads | ~1 min |


In [ ]:
# 1. Install (the versions Colab already has for torch are kept)
%pip install -q -U "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.45" "datasets>=3.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
# 2. Setup: GPU, Google Drive, data, settings
import os, sys, json, re, time, glob, shutil, inspect
import torch
import pandas as pd

TEST = os.environ.get("C4_TEST") == "1"          # offline dry-run only; leave as is
IN_COLAB = "google.colab" in sys.modules
if not TEST:
    assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU > Save, then Run all again."
    print("GPU:", torch.cuda.get_device_name(0))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/c4_finetune"
else:
    WORK = os.environ.get("C4_WORK", "c4_finetune")
os.makedirs(WORK, exist_ok=True)

def _have_data():
    return all(os.path.exists(f"{WORK}/{n}") for n in ("train.jsonl", "val.jsonl"))

if not _have_data() and IN_COLAB:
    from google.colab import files
    print("Select BOTH train.jsonl and val.jsonl")
    for name, data in files.upload().items():
        target = "train.jsonl" if "train" in name else "val.jsonl" if "val" in name else None
        if target:
            open(f"{WORK}/{target}", "wb").write(data)
assert _have_data(), f"train.jsonl and val.jsonl must be in {WORK}"

MODEL_ID = os.environ.get("C4_MODEL", "unsloth/mistral-7b-instruct-v0.3")   # open copy of mistralai/Mistral-7B-Instruct-v0.3
MAX_LEN = 1536        # tokens per example (prompt + answer are ~900-1200)
MAX_NEW = 700         # 3 emails of <150 words + JSON (set num_predict >= this in emailAgent.js)
N_EVAL = 3 if TEST else 25
EPOCHS, LR, LORA_R, SEED = 2, 2e-4, 16, 42
torch.manual_seed(SEED)
print("Work folder:", WORK)
print({n: sum(1 for _ in open(f"{WORK}/{n}", encoding="utf-8")) for n in ("train.jsonl", "val.jsonl")})

GPU: Tesla T4
Mounted at /content/drive
Work folder: /content/drive/MyDrive/c4_finetune
{'train.jsonl': 715, 'val.jsonl': 171}


In [ ]:
# 3. Load the tokenizer and Mistral 7B in 4-bit (QLoRA base)
import transformers
from packaging import version
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print("transformers", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if not tokenizer.chat_template:                      # official Mistral v0.3 template, in case the repo has none
    tokenizer.chat_template = ("{{ bos_token }}{% for m in messages %}{% if m['role'] == 'user' %}"
                               "{{ '[INST] ' + m['content'] + '[/INST]' }}{% elif m['role'] == 'assistant' %}"
                               "{{ ' ' + m['content'] + eos_token }}{% endif %}{% endfor %}")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token
tokenizer.padding_side = "right"

# non-quantised parts (embeddings, norms, lm_head) in float32: the T4 has no bfloat16
dtype_kw = {"dtype": torch.float32} if version.parse(transformers.__version__) >= version.parse("4.56") \
    else {"torch_dtype": torch.float32}
if TEST:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **dtype_kw)
else:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0}, **dtype_kw)
model.config.use_cache = True
print(f"model loaded: {model.get_memory_footprint() / 1e9:.1f} GB")

transformers 5.20.2


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

model loaded: 4.6 GB


In [ ]:
# 4. Tokenise: the loss is computed on the answer only (prompt tokens are masked with -100)
from datasets import load_dataset
raw = load_dataset("json", data_files={"train": f"{WORK}/train.jsonl", "validation": f"{WORK}/val.jsonl"})

def encode(ex):
    msgs = ex["messages"]
    prompt = tokenizer.apply_chat_template(msgs[:1], tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(msgs, tokenize=False)
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    return {"input_ids": f_ids, "attention_mask": [1] * len(f_ids),
            "labels": [-100] * len(p_ids) + f_ids[len(p_ids):],
            "prefix_ok": f_ids[:len(p_ids)] == p_ids, "n_tokens": len(f_ids)}

tok = raw.map(encode, remove_columns=["messages"])
for split in ("train", "validation"):
    t = tok[split]
    assert all(t["prefix_ok"]), "prompt/answer split failed - send this message to Claude"
    print(split, len(t), "examples | tokens median", int(pd.Series(t["n_tokens"]).median()), "max", max(t["n_tokens"]))
tok = tok.filter(lambda e: e["n_tokens"] <= MAX_LEN).remove_columns(["prefix_ok", "n_tokens"])
print("after length filter:", {k: len(v) for k, v in tok.items()})

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/715 [00:00<?, ? examples/s]

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

train 715 examples | tokens median 923 max 1230
validation 171 examples | tokens median 922 max 1115


Filter:   0%|          | 0/715 [00:00<?, ? examples/s]

Filter:   0%|          | 0/171 [00:00<?, ? examples/s]

after length filter: {'train': 715, 'validation': 171}


In [ ]:
# 5. Evaluation: the same checks as guardrails.js + the grounding check, on the validation prompts
UNSUB = "To unsubscribe from these emails, reply STOP or click here."
PII_RE = re.compile(r"\b(\+?\d[\d\s\-]{7,}\d|[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}|\d{9,12})\b")
FACT_RE = re.compile(r"\d+(?:\.\d+)?%|\$\s?\d[\d,]*(?:\.\d+)?|\b\d+x\b")
CODE_RE = re.compile(r"\b(?:[Ss]ave|SAVE|CODE|PROMO|DEAL|OFF)\d+\b|\b[A-Z]{3,}\d{1,3}\b")
SPAM = ["free!!!", "guaranteed income", "act now!!!", "limited time offer", "click here now", "make money fast"]

def check(prompt, text):
    context = prompt.split("You are an email marketing specialist.")[0] + " " + \
              (re.search(r"Campaign goal: (.*)", prompt) or [None, ""])[1]
    no_offer = "There is no discount or promo code" in prompt
    empty = {"json_valid": False, "three_emails": False, "unsubscribe_all": False, "grounded": False,
             "no_invented_offer": False, "under_150_words": False, "guardrails_pass": False}
    try:
        out = json.loads(text[text.index("{"): text.rindex("}") + 1])
        emails = out["emails"]
        assert isinstance(emails, list) and all(isinstance(e, dict) for e in emails)
        for e in emails:
            e["subject"], e["body"] = str(e.get("subject", "")), str(e.get("body", ""))
    except Exception:
        return empty
    r = {"json_valid": True}
    r["three_emails"] = len(emails) == 3 and [e.get("send_at_days") for e in emails] == [0, 3, 7]
    r["unsubscribe_all"] = all(re.search(r"(?i)unsubscribe|opt.?out|reply\s+stop", e["body"]) for e in emails)
    facts = [f for e in emails for f in FACT_RE.findall(e["subject"] + " " + e["body"])]
    codes = [c for e in emails for c in CODE_RE.findall(e["subject"] + " " + e["body"]) if c != "STOP"]
    r["grounded"] = all(f.replace(" ", "") in context.replace(" ", "") for f in facts) and all(c in context for c in codes)
    r["no_invented_offer"] = (not facts and not codes) if no_offer else True
    r["under_150_words"] = all(len(e["body"].split()) <= 150 for e in emails)
    pii = any(PII_RE.search(e["subject"] + " " + e["body"]) for e in emails)
    spam = any(w in (e["subject"] + " " + e["body"]).lower() for e in emails for w in SPAM)
    caps = any(len(re.findall(r"[A-Z]{4,}", (e["subject"] + " " + e["body"]).replace("STOP", ""))) > 3 for e in emails)
    r["guardrails_pass"] = r["unsubscribe_all"] and r["grounded"] and not (pii or spam or caps)
    return r

@torch.no_grad()
def generate(prompt):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, add_special_tokens=False, return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

def evaluate(tag):
    path = f"{WORK}/c4-ft_eval_{tag}.csv"
    if os.path.exists(path):                       # already done before a disconnect
        print("loaded saved", path)
        return pd.read_csv(path)
    model.eval()
    model.config.use_cache = True
    val = raw["validation"].shuffle(seed=SEED).select(range(min(N_EVAL, len(raw["validation"]))))
    rows = []
    for i, ex in enumerate(val):
        prompt = ex["messages"][0]["content"]
        t0 = time.time()
        text = generate(prompt)
        rows.append({"i": i, **check(prompt, text), "seconds": round(time.time() - t0, 2), "model": tag,
                     "output": text[:1500]})
        print(f"{tag}: {i + 1}/{len(val)}", end="\r")
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    print("\n", tag, df.drop(columns=["i"]).mean(numeric_only=True).round(3).to_dict())
    return df

# sanity check: the reference answers must pass every check
ref = [check(e["messages"][0]["content"], e["messages"][1]["content"]) for e in raw["validation"]]
print("reference answers pass rate:", pd.DataFrame(ref).mean().round(3).to_dict())


reference answers pass rate: {'json_valid': 1.0, 'three_emails': 1.0, 'unsubscribe_all': 1.0, 'grounded': 1.0, 'no_invented_offer': 1.0, 'under_150_words': 1.0, 'guardrails_pass': 1.0}


In [ ]:
# 6. BEFORE fine-tuning: base Mistral 7B on the validation prompts
before = evaluate("base")
print(before.iloc[0]["output"])

[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


base: 25/25
 base {'json_valid': 1.0, 'three_emails': 1.0, 'unsubscribe_all': 0.32, 'grounded': 0.92, 'no_invented_offer': 1.0, 'under_150_words': 1.0, 'guardrails_pass': 0.24, 'seconds': 31.074}
{
  "emails": [
    {
      "email_index": 1,
      "subject": "Welcome back to Sunday Paper Press, {{first_name}}!",
      "body": "We're excited to share our latest blog post with you. This week, we're diving into the fascinating world of sustainable fashion. Discover how you can make a difference with your wardrobe choices. Click here to read more: {{cta_link}}",
      "send_at_days": 0
    },
    {
      "email_index": 2,
      "subject": "Your Sustainable Fashion Journey Awaits, {{first_name}}!",
      "body": "We hope you enjoyed our latest blog post! Today, we're offering a special 25% off on all fashion-related items as a token of our appreciation. Use promo code JOIN25 on orders over $40. Offer ends in 48 hours. Click here to shop now: {{cta_link}}",
      "send_at_days": 3
    },
   

In [ ]:
# 7. QLoRA training (resumes from the last checkpoint after a disconnect)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from transformers.trainer_utils import get_last_checkpoint

ADAPTER_DIR = f"{WORK}/c4-email-lora"
if os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"):
    from peft import PeftModel
    print("adapter already trained - loading it from Drive")
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    trainer = None
else:
    if not TEST:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()

    steps = -(-len(tok["train"]) // 8) * EPOCHS              # optimiser steps (batch 1 x accumulation 8)
    kw = dict(output_dir=f"{WORK}/checkpoints", num_train_epochs=EPOCHS, learning_rate=LR,
              per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8,
              lr_scheduler_type="cosine", warmup_steps=max(1, steps // 20), logging_steps=5,
              eval_steps=20, save_strategy="steps", save_steps=20, save_total_limit=2,
              fp16=not TEST, optim="adamw_torch" if TEST else "paged_adamw_8bit",
              report_to="none", seed=SEED)
    if TEST:
        kw.update(max_steps=4, eval_steps=2, save_steps=2, logging_steps=1)
    sig = inspect.signature(TrainingArguments.__init__).parameters
    kw["eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"] = "steps"
    dropped = [k for k in kw if k not in sig]                # settings this transformers version does not know
    if dropped:
        print("skipping settings not supported by this transformers version:", dropped)
    args = TrainingArguments(**{k: v for k, v in kw.items() if k in sig})
    print(f"training: {len(tok['train'])} examples, {steps} optimiser steps")
    trainer = Trainer(model=model, args=args, train_dataset=tok["train"], eval_dataset=tok["validation"],
                      data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
    last = get_last_checkpoint(args.output_dir) if os.path.isdir(args.output_dir) else None
    print("resuming from", last) if last else print("starting training")
    trainer.train(resume_from_checkpoint=last)
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    pd.DataFrame(trainer.state.log_history).to_csv(f"{WORK}/c4-ft_training_log.csv", index=False)
    print("adapter saved to", ADAPTER_DIR)

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754
training: 715 examples, 180 optimiser steps
starting training


Step,Training Loss,Validation Loss
20,0.658101,0.879255
40,0.473109,0.870928
60,0.410661,0.883106
80,0.357343,0.876252
100,0.194860,0.980244
120,0.156704,0.971742
140,0.187865,0.982837
160,0.131630,0.990613
180,0.129614,0.992434


adapter saved to /content/drive/MyDrive/c4_finetune/c4-email-lora


In [ ]:
# 8. AFTER fine-tuning: same validation prompts, then the comparison table
after = evaluate("fine-tuned")
cols = ["json_valid", "three_emails", "unsubscribe_all", "grounded", "no_invented_offer",
        "under_150_words", "guardrails_pass", "seconds"]
table = pd.concat([before, after]).groupby("model")[cols].mean().round(3).reindex(["base", "fine-tuned"])
print(table)
table.to_csv(f"{WORK}/c4-ft_finetune_eval.csv")
summary = {"base_model": MODEL_ID, "method": "QLoRA 4-bit nf4, LoRA r=%d alpha=%d, %d epochs, lr %g" % (LORA_R, 2 * LORA_R, EPOCHS, LR),
           "train_examples": len(tok["train"]), "eval_prompts": len(after),
           "before": table.loc["base"].to_dict(), "after": table.loc["fine-tuned"].to_dict()}
log_path = f"{WORK}/c4-ft_training_log.csv"
if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    if "loss" in log:
        summary["train_loss_first_last"] = [round(float(log["loss"].dropna().iloc[0]), 4), round(float(log["loss"].dropna().iloc[-1]), 4)]
    if "eval_loss" in log:
        summary["eval_loss_first_last"] = [round(float(log["eval_loss"].dropna().iloc[0]), 4), round(float(log["eval_loss"].dropna().iloc[-1]), 4)]
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        ax = log.dropna(subset=["loss"]).plot(x="step", y="loss", label="train loss")
        if "eval_loss" in log:
            log.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", label="validation loss", ax=ax)
        ax.set_title("C4 email QLoRA training loss"); ax.set_xlabel("step")
        plt.savefig(f"{WORK}/c4-ft_loss.png", dpi=120, bbox_inches="tight")
    except Exception as e:
        print("loss plot skipped:", e)
json.dump(summary, open(f"{WORK}/c4-ft_finetune_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


fine-tuned: 25/25
 fine-tuned {'json_valid': 0.96, 'three_emails': 0.96, 'unsubscribe_all': 0.96, 'grounded': 0.76, 'no_invented_offer': 0.8, 'under_150_words': 0.96, 'guardrails_pass': 0.76, 'seconds': 64.28}
            json_valid  three_emails  unsubscribe_all  grounded  \
model                                                             
base              1.00          1.00             0.32      0.92   
fine-tuned        0.96          0.96             0.96      0.76   

            no_invented_offer  under_150_words  guardrails_pass  seconds  
model                                                                     
base                      1.0             1.00             0.24   31.074  
fine-tuned                0.8             0.96             0.76   64.280  
{
  "base_model": "unsloth/mistral-7b-instruct-v0.3",
  "method": "QLoRA 4-bit nf4, LoRA r=16 alpha=32, 2 epochs, lr 0.0002",
  "train_examples": 715,
  "eval_prompts": 25,
  "before": {
    "json_valid": 1.0,
    "three_

In [ ]:
# 9. Package the adapter and download the results (everything is also in Drive: MyDrive/c4_finetune)
shutil.make_archive(f"{WORK}/c4-email-lora", "zip", f"{WORK}/c4-email-lora")
for f in sorted(glob.glob(f"{WORK}/c4-ft_*")) + [f"{WORK}/c4-email-lora.zip"]:
    print(f"{os.path.getsize(f) / 1e6:8.1f} MB  {f}")
if IN_COLAB:
    from google.colab import files
    for f in ["c4-ft_finetune_eval.csv", "c4-ft_finetune_summary.json", "c4-ft_training_log.csv",
              "c4-ft_eval_base.csv", "c4-ft_eval_fine-tuned.csv", "c4-ft_loss.png", "c4-email-lora.zip"]:
        if os.path.exists(f"{WORK}/{f}"):
            files.download(f"{WORK}/{f}")
print("Next: File > Download > Download .ipynb  (save as services/email-agent/ml/finetune/c4_finetune_mistral_qlora.ipynb)")

     0.0 MB  /content/drive/MyDrive/c4_finetune/c4-ft_eval_base.csv
     0.0 MB  /content/drive/MyDrive/c4_finetune/c4-ft_eval_fine-tuned.csv
     0.0 MB  /content/drive/MyDrive/c4_finetune/c4-ft_finetune_eval.csv
     0.0 MB  /content/drive/MyDrive/c4_finetune/c4-ft_finetune_summary.json
     0.0 MB  /content/drive/MyDrive/c4_finetune/c4-ft_loss.png
     0.0 MB  /content/drive/MyDrive/c4_finetune/c4-ft_training_log.csv
   156.1 MB  /content/drive/MyDrive/c4_finetune/c4-email-lora.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Next: File > Download > Download .ipynb  (save as services/email-agent/ml/finetune/c4_finetune_mistral_qlora.ipynb)
